# 🏦 Magentic Multi-Agent Orchestration for Investment Research

## Overview

This notebook demonstrates **Magentic orchestration** for financial services - an advanced multi-agent coordination pattern for comprehensive investment research and analysis. The orchestrator automatically manages research task decomposition, specialist agent selection, and insight synthesis.

> ⚠️ **Financial Services Disclaimer**: This notebook demonstrates AI agent workflows for educational purposes. In production, all investment research and recommendations must comply with regulatory requirements (SEC, FINRA) and include appropriate disclosures.

### Industry Use Case: Investment Research Report

A wealth management firm needs to prepare quarterly investment research reports that require:
- **Market Research**: Current market conditions, sector trends
- **Quantitative Analysis**: Financial calculations, risk metrics
- **Report Synthesis**: Consolidated insights and recommendations

### Key Concepts

| Concept | Description |
|---------|-------------|
| **MagenticBuilder** | High-level API for multi-agent orchestration |
| **Standard Manager** | Built-in orchestrator with planning capabilities |
| **Specialized Agents** | Domain experts (Market Researcher, Quant Analyst) |
| **Streaming Callbacks** | Real-time event monitoring |
| **Event Types** | Orchestrator messages, agent deltas, agent messages, final result |

### Architecture

```
Investment Research Request
           ↓
Magentic Orchestrator (Standard Manager)
           ↓
Research Plan Generation
           ↓
┌─────────────────────────────────┐
│     Delegate to Specialists     │
│  ├── MarketResearcherAgent      │
│  │   (Market data, sector news) │
│  └── QuantAnalystAgent          │
│      (Financial calculations)   │
└─────────────────────────────────┘
           ↓
Monitor & Adapt Plan
           ↓
Synthesize Investment Report
```

### Workflow Parameters

| Parameter | Default | Purpose |
|-----------|---------|---------|
| `max_round_count` | 10 | Maximum orchestration rounds |
| `max_stall_count` | 3 | Retries when progress stalls |
| `max_reset_count` | 2 | Full plan resets allowed |

## Prerequisites

- ✅ Azure AI Foundry configured
- ✅ Environment variables: `AI_FOUNDRY_PROJECT_ENDPOINT`, `AZURE_AI_MODEL_DEPLOYMENT_NAME`
- ✅ Azure CLI authentication: Run `az login`

## 1️⃣ Setup and Imports

In [ ]:
import asyncio
import json
import os
from dotenv import load_dotenv
from typing import cast

from azure.identity import AzureCliCredential

from agent_framework import (
    Agent,
    AgentResponse,
    AgentResponseUpdate,
    Message,
    WorkflowEvent,
)
from agent_framework.orchestrations import (
    MagenticBuilder,
    MagenticOrchestratorEvent,
    MagenticProgressLedger,
)
from agent_framework.foundry import FoundryChatClient

# Load environment variables from .env file
load_dotenv('../../.env')

# Verify environment is loaded
project_endpoint = os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")
print(f"✅ Environment loaded: {project_endpoint is not None and model_deployment is not None}")

## 2️⃣ Create Foundry Chat Client

We'll use a single Foundry client for all agents to share credentials and endpoint configuration.

In [ ]:
# Create shared Foundry Chat Client
chat_client = FoundryChatClient(
    credential=AzureCliCredential(),
    project_endpoint=os.getenv("AI_FOUNDRY_PROJECT_ENDPOINT"),
    model=os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME"),
)

print("✅ Foundry Chat Client created")

## 3️⃣ Create Specialized Financial Agents

### MarketResearcherAgent
- Specializes in market data, sector trends, and financial news
- Gathers factual information about market conditions

### QuantAnalystAgent
- Performs financial calculations and risk analysis
- Provides detailed analysis with risk metrics and projections

In [ ]:
# Create Market Researcher Agent
market_researcher = Agent(
    client=chat_client,
    name="MarketResearcherAgent",
    description="Specialist in market research, sector analysis, and financial news gathering",
    instructions=(
        "You are a Financial Market Researcher at a wealth management firm. "
        "You gather market data, sector trends, company news, and economic indicators. "
        "Focus on factual information without performing calculations. "
        "Always cite sources and note the date of any market data you reference."
    ),
)

# Create Quantitative Analyst Agent
quant_analyst = Agent(
    client=chat_client,
    name="QuantAnalystAgent",
    description="A quantitative analyst that performs financial calculations, risk analysis, and portfolio modeling.",
    instructions=(
        "You are a Quantitative Analyst at a wealth management firm. "
        "You perform financial calculations including: portfolio analysis, risk metrics (beta, Sharpe ratio, VaR), "
        "return calculations, and investment projections. "
        "Always show your calculation methodology and assumptions. "
        "Present results in clear tables when appropriate."
    ),
)

# Create Manager Agent to orchestrate the workflow
manager_agent = Agent(
    client=chat_client,
    name="ResearchManager",
    description="Orchestrator that coordinates the investment research workflow",
    instructions=(
        "You coordinate a team of financial analysts to complete investment research tasks efficiently. "
        "Review outputs from the Market Researcher and pass relevant findings to the Quant Analyst. "
        "Synthesize a final comprehensive investment research report. "
        "Ensure all research includes appropriate risk disclosures."
    ),
)

print("✅ Market Researcher Agent created")
print("✅ Quant Analyst Agent created")
print("✅ Manager Agent created")

## 4️⃣ Build Magentic Investment Research Workflow

### Configuration

The `MagenticBuilder` sets up a dynamic multi-agent orchestration where the manager coordinates specialized agents based on task requirements.

| Parameter | Purpose |
|-----------|---------|
| `participants` | Specialist agents available to the manager |
| `intermediate_output_from` | Agents whose outputs stream in real-time |
| `manager_agent` | The orchestrator agent that plans and delegates |
| `max_round_count` | Maximum coordination rounds |
| `max_stall_count` | Retries when progress stalls |
| `max_reset_count` | Full plan resets allowed |

In [ ]:
print("\n🏦 Building Magentic Investment Research Workflow...")

# Build the Magentic workflow with dynamic agent orchestration
workflow = MagenticBuilder(
    participants=[market_researcher, quant_analyst],
    intermediate_output_from=[market_researcher, quant_analyst],
    manager_agent=manager_agent,
    max_round_count=10,
    max_stall_count=3,
    max_reset_count=2,
).build()

print("✅ Magentic investment research workflow built successfully")
print("\nWorkflow Configuration:")
print("  • Participants: MarketResearcherAgent, QuantAnalystAgent")
print("  • Manager: ResearchManager (dynamic orchestration)")
print("  • Intermediate Output: Enabled for both participants")
print("  • Max Rounds: 10 | Max Stalls: 3 | Max Resets: 2")

## 5️⃣ Define Investment Research Task

This quarterly research request requires:

1. **Market Research**: Current technology sector trends and market conditions
2. **Quantitative Analysis**: Calculate risk-adjusted returns and portfolio metrics
3. **Comparison**: Evaluate different investment approaches
4. **Synthesis**: Provide actionable recommendations with risk assessment

The orchestrator will automatically decompose this into subtasks and delegate to the appropriate specialists.

In [ ]:
# Investment research request from a wealth advisor
research_request = (
    "I need a quarterly investment research brief for a client considering technology sector exposure. "
    "Please research the current state of the technology sector including recent performance of major indices "
    "(NASDAQ-100, S&P Technology Select Sector) and key market drivers. "
    "Then calculate the risk-adjusted return metrics for a hypothetical $100,000 investment allocated as: "
    "40% large-cap tech ETF (assume 12% annual return, 18% volatility), "
    "35% mid-cap growth fund (assume 15% annual return, 25% volatility), "
    "and 25% dividend tech stocks (assume 8% annual return, 12% volatility). "
    "Provide a summary table comparing expected returns, portfolio volatility, and Sharpe ratio "
    "(assuming 5% risk-free rate). Include a recommendation with risk considerations."
)

print(f"📋 Research Request: {research_request}")

## 6️⃣ Execute Research Workflow with Magentic Streaming

The Magentic orchestration follows this execution pattern:
1. **Planning Phase**: The manager analyzes the task and creates a research plan
2. **Agent Selection**: Manager dynamically selects the most appropriate agent per subtask
3. **Execution**: Selected agent executes their portion (streams intermediate output)
4. **Progress Assessment**: Manager evaluates progress and updates the plan
5. **Stall Detection**: If progress stalls, auto-replan is triggered
6. **Final Synthesis**: Manager synthesizes all agent outputs into a final research report

### Event Types

| Event Type | Description |
|------------|-------------|
| `AgentResponseUpdate` | Streaming text from participant agents |
| `magentic_orchestrator` | Plan created, replanned, or progress ledger updated |
| `AgentResponse` | Final synthesized answer from the manager |

In [ ]:
async def run_investment_research():
    """Run the Magentic investment research workflow with streaming output."""
    
    print("\n🚀 Starting Magentic investment research workflow...")
    print("=" * 60)
    
    # Track streaming state
    last_message_id: str | None = None
    final_response: AgentResponse | None = None

    async for event in workflow.run(research_request, stream=True):
        # Stream intermediate agent outputs (from participants)
        if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
            message_id = event.data.message_id
            if message_id != last_message_id:
                if last_message_id is not None:
                    print("\n")
                print(f"\n- {event.executor_id}:", end=" ", flush=True)
                last_message_id = message_id
            print(event.data, end="", flush=True)

        # Handle Magentic orchestrator events (plan created, replanned, progress ledger)
        elif event.type == "magentic_orchestrator":
            print(f"\n\n[🎯 Magentic Orchestrator] {event.data.event_type.name}")
            if isinstance(event.data.content, Message):
                print(f"📋 Plan:\n{event.data.content.text}")
            elif isinstance(event.data.content, MagenticProgressLedger):
                print(f"📊 Progress Ledger:\n{json.dumps(event.data.content.to_dict(), indent=2)}")

        # Capture final synthesized response from the manager
        elif event.type == "output" and isinstance(event.data, AgentResponse):
            final_response = event.data

    # Display the manager's final synthesized answer
    print("\n\n" + "=" * 60)
    print("📊 INVESTMENT RESEARCH REPORT COMPLETE")
    print("=" * 60)
    
    if final_response and final_response.messages:
        print(final_response.messages[-1].text)
    
    print("\n" + "=" * 60)
    print("⚠️ DISCLAIMER: This is for demonstration purposes only.")
    print("   Actual investment decisions require professional advice.")

## 7️⃣ Run the Investment Research Workflow

In [ ]:
await run_investment_research()

## 📝 Key Takeaways

### Magentic for Financial Research

| Benefit | Description |
|---------|-------------|
| **Intelligent Planning** | Orchestrator decomposes complex research requests |
| **Specialist Delegation** | Market researcher vs. quantitative analyst |
| **Adaptive Execution** | Adjusts plan based on findings |
| **Synthesized Reports** | Consolidated insights from multiple specialists |

### Industry Use Cases for Magentic Orchestration

| Use Case | Specialists Needed |
|----------|-------------------|
| Investment Research | Market Researcher + Quant Analyst |
| Due Diligence | Legal Analyst + Financial Analyst |
| Risk Assessment | Risk Analyst + Market Researcher |
| Client Reporting | Data Analyst + Compliance Writer |

### Magentic vs. Other Orchestration Patterns

| Pattern | When to Use | Key Feature |
|---------|-------------|-------------|
| **Concurrent** | Independent parallel tasks | Fan-out/fan-in |
| **Sequential** | Linear, dependent steps | Shared context |
| **Magentic** | Complex, adaptive decomposition | Intelligent orchestration |


### Production Considerations for FSI

| Consideration | Recommendation |
|---------------|----------------|
| **Compliance** | Log all orchestrator decisions for audit trail |
| **Cost Control** | Set appropriate max_round_count |
| **Data Currency** | Timestamp all market data references |
| **Disclaimers** | Include regulatory disclosures in final output |
| **Review** | Human review required before client delivery |